In [1]:
import numpy as np
from glob import glob
from DNN.ult_spacenet import SeismicDataset, SeismicTrainer, create_data_loaders, RGTNetwork3D

In [2]:
def normalize_seismic(seismic):
    """Normalize seismic to [0, 1] using min-max."""
    min_val = np.min(seismic)
    max_val = np.max(seismic)
    if max_val - min_val == 0:
        return np.zeros_like(seismic)
    return (seismic - min_val) / (max_val - min_val)

def normalize_rgt(rgt):
    """Normalize RGT by mean and std."""
    mean_val = np.mean(rgt)
    std_val = np.std(rgt)
    if std_val == 0:
        return np.zeros_like(rgt)
    normalized = (rgt - mean_val) / std_val
    # Clamp values to [-1, 1] after standardization
    return normalized

def slice_data_chunks_with_stride(data, chunk_size=(128, 128, 128), stride=None):
    """
    Slice input data of shape (150, 150, 2000) into overlapping chunks of specified size.
    
    Args:
        data: Input numpy array of shape (150, 150, 2000)
        chunk_size: Tuple of (height, width, depth) for each chunk
        stride: Step size between chunks (default is half the chunk size for 50% overlap)
    
    Returns:
        List of numpy arrays, each of shape chunk_size
    """
    h, w, d = data.shape
    ch, cw, cd = chunk_size
    
    # Default stride is half the chunk size for 50% overlap
    if stride is None:
        stride = (ch // 2, cw // 2, cd // 2)
    
    sh, sw, sd = stride
    
    chunks = []
    
    # Calculate how many chunks we can fit in each dimension with the given stride
    h_steps = max(1, (h - ch) // sh + 1)
    w_steps = max(1, (w - cw) // sw + 1)
    d_steps = max(1, (d - cd) // sd + 1)
    
    for i in range(h_steps):
        for j in range(w_steps):
            for k in range(d_steps):
                start_h, end_h = i * sh, i * sh + ch
                start_w, end_w = j * sw, j * sw + cw
                start_d, end_d = k * sd, k * sd + cd
                
                # Ensure we don't exceed the data bounds
                end_h = min(end_h, h)
                end_w = min(end_w, w)
                end_d = min(end_d, d)
                
                # Adjust start positions if we're near the boundary
                start_h = max(0, end_h - ch)
                start_w = max(0, end_w - cw)
                start_d = max(0, end_d - cd)
                
                chunk = data[start_h:end_h, start_w:end_w, start_d:end_d]
                chunks.append(chunk)
    del chunks[0:3]
    return chunks

In [3]:
seismic = glob("../data/synthetic_data/run/*150-150-2000*/seismicCubes_cumsum_fullstack*.npy")
age = glob("../data/synthetic_data/run/*150-150-2000*/faulted_age*.npy")
seis_chunk= []
age_chunk = []
for s, a in zip(seismic, age):
    sdf = np.load(s)
    adf = np.load(a)
    seis_chunk.extend(slice_data_chunks_with_stride(sdf))
    age_chunk.extend(slice_data_chunks_with_stride(adf))
age_chunk_norm = []
seis_chunk_norm = []
for i in range(len(age_chunk)):
    age_chunk_norm.append(normalize_rgt(age_chunk[i]))
    seis_chunk_norm.append(normalize_seismic(seis_chunk[i]))

In [4]:
# from util.plotting import plot_3d_array_interactive
# plot_3d_array_interactive(age_chunk_norm[2], axis='x')
# plot_3d_array_interactive(seis_chunk_norm[2], axis='x')

In [5]:
dataset = SeismicDataset(seis_chunk_norm, age_chunk_norm)
train_loader, val_loader = create_data_loaders(dataset, batch_size=2)
model = RGTNetwork3D(input_size=128)
trainer = SeismicTrainer(model, train_loader, val_loader, device='cuda')

In [6]:
trainer.train(20, "../data/DNN models")

RuntimeError: Given groups=1, weight of size [1024, 1024, 1, 1, 1], expected input[2, 2048, 4, 4, 4] to have 1024 channels, but got 2048 channels instead